# 06｜历史相似行情与 2026 敏感度机制分析

本 Notebook 是诊断分析入口，不修改冻结运行包的信号、阈值、持有参数或输出格式。

它读取：

- 本地米筐 CSI500 现货；
- 03 已生成的含持有期精简八列表；
- 事件型八列表和冻结参考，用于逐列一致性审计；
- 冻结 1545 内部面板，用于机制诊断。

输出包括历史相似行情、市场环境与内部状态分解、多视角同步与确认诊断、固定开发期锚定代理、相似阶段未来表现和本地/远端一致性审计。历史窗口的排序只使用窗口结束时已经可见的信息，后续收益只作为事后评价。

In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = Path(os.environ.get(
    'FINAL_UPLOAD_PACKAGE_ROOT',
    '/home/hzy/cta/20260824_1325_冻结中证500输出包',
)).expanduser().resolve()
SRC_ROOT = (PACKAGE_ROOT / 'src').resolve()
if not PACKAGE_ROOT.is_absolute() or not SRC_ROOT.is_absolute():
    raise RuntimeError('06 的包路径必须是绝对路径')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

def absolute_env(name, default):
    value = os.environ.get(name, default).strip()
    path = Path(value).expanduser().resolve()
    if not path.is_absolute():
        raise RuntimeError(f'{name} 必须是绝对路径：{value}')
    return path

SPOT_PATH = absolute_env('COMPANY_SPOT_PATH', '/home/hzy/cta/IC数据更新_最终固化版/现货最终版/CSI500_SPOT_md_eod_raw_最终版.parquet')
HOLDING_PATH = absolute_env('HOLDING_EIGHT_PATH', str(PACKAGE_ROOT / 'runtime_outputs_holding_period' / '含持有期八列表.csv'))
EVENT_PATH = absolute_env('EVENT_EIGHT_PATH', str(PACKAGE_ROOT / 'runtime_outputs' / '最终执行日简表.csv'))
EXPECTED_PATH = absolute_env('EXPECTED_EIGHT_PATH', str(PACKAGE_ROOT / '本地结果对照/01_只用现货生成八列/最终执行日简表.csv'))
STAGE04_DIR = absolute_env('ANALYSIS_04_OUTPUT_DIR', str(PACKAGE_ROOT / 'runtime_outputs_04_returns'))
OUTPUT_DIR = absolute_env('ANALYSIS_06_OUTPUT_DIR', str(PACKAGE_ROOT / 'runtime_outputs_06_mechanism'))
PANEL_TEXT = os.environ.get('PANEL_PATH', '').strip()
PANEL_PATH = Path(PANEL_TEXT).expanduser().resolve() if PANEL_TEXT else None
if PANEL_PATH is not None and not PANEL_PATH.is_absolute():
    raise RuntimeError('PANEL_PATH 必须是绝对路径')

from reproduce_stage_06 import run_stage_06

print('包目录：', PACKAGE_ROOT)
print('本地米筐现货：', SPOT_PATH)
print('事件型八列表：', EVENT_PATH)
print('含持有期八列表：', HOLDING_PATH)
print('06 输出目录：', OUTPUT_DIR)

## 1. 运行 06 分析

事件型八列和含持有期八列在审计中分开处理：前者与冻结参考逐列比较，后者只用于连续持有路径和收益分析。

In [ ]:
metadata = run_stage_06(
    SPOT_PATH,
    HOLDING_PATH,
    OUTPUT_DIR,
    EXPECTED_PATH,
    STAGE04_DIR,
    PANEL_PATH,
    EVENT_PATH,
)
print(json.dumps(metadata['audit'], ensure_ascii=False, indent=2))
print('历史相似窗口：', metadata['analogue_dates'])
print('图像数量：', len(metadata['figures']))
print('表格数量：', len(metadata['tables']))

## 2. 关键表格

下面的指标只用于诊断和解释，不会回写冻结参数。

In [ ]:
tables = OUTPUT_DIR / 'tables'
audit = pd.read_csv(tables / '本地远端一致性审计摘要.csv', encoding='utf-8-sig')
analogs = pd.read_csv(tables / '历史相似行情候选_60日.csv', encoding='utf-8-sig')
annual = pd.read_csv(tables / '年度行情机制与收益对比.csv', encoding='utf-8-sig')
display(audit)
display(analogs)
display(annual.tail(8))

## 3. 解释边界

图 01—02 用历史相似窗口回答“以前有没有类似行情”；图 03—05 分解市场环境、滚动相对分数、同步性和确认/滞回；图 06—07 比较相似阶段的后续表现及年度覆盖变化。

固定开发期锚定是诊断代理，不是正式冻结逻辑。要完全拆出每个原始因子进入滚动 z-score 前后的反事实，需要额外保存冻结引擎的原始中间特征，但本 Notebook 不因此修改生产引擎。

## 4. 老师红色0区间的同口径逐段复盘

这一节对应老师图片中标出的三个红色区间。窗口按图片坐标固定为：2023-11-01—2024-01-15、2024-04-15—2024-09-15、2025-03-01—2025-07-15。三条曲线都使用04相同的执行日O2O加算口径：执行日开盘到下一实际交易日开盘，NAV=1+累计加算收益，不复利。局部图仅把窗口首日NAV重新设为1，便于比较。

In [ ]:
from IPython.display import Image

teacher_summary = pd.read_csv(OUTPUT_DIR / 'tables/老师红色区间复盘汇总_O2O.csv', encoding='utf-8-sig')
display(teacher_summary)
display(Image(filename=str(OUTPUT_DIR / 'figures/28_老师红色区间_三状态与指数_O2O局部复盘.png')))
for figure_name in [
    '28A_红色区间1_三状态与指数_O2O局部复盘.png',
    '28B_红色区间2_三状态与指数_O2O局部复盘.png',
    '28C_红色区间3_三状态与指数_O2O局部复盘.png',
]:
    display(Image(filename=str(OUTPUT_DIR / 'figures' / figure_name)))